# 딥러닝응용I(추천시스템)
**동덕여자대학교 데이터사이언스전공 · 유원상 교수 · 2026년 2학기**

## W02A · 기본적인 추천 방법 (1)
데이터 준비, 인기 기준 비교, 집단별 추천, 기본 평가를 거쳐 지난 앱을 확장한다.
공통 함수·클래스는 `luna_recsys` 패키지에 누적한다.

**진행 방법:** 위에서부터 실행한다. 자동 데이터 준비 셀에서 실제 모드를 확인한다.
중간 `None`은 직접 완성할 문제이다. 미완성 안내가 나와도 다음 독립 실습은 실행할 수 있다.
오류 예제는 오류를 관찰하는 코드와 고치는 코드를 구분한다. 출력은 저장 전에 지운다.


## 1. 검토한 수업 코드 준비
Colab에서는 해당 차시의 공개 버전에서 코드를 설치한다. `sys.executable`은 현재
notebook의 Python이다. 설치가 실패하면 버전 공개 여부와 네트워크를 확인하고 계속 실행하지 않는다.
로컬 검증에서는 이미 설치한 저작 패키지를 사용한다.
최초 공개 전 이 notebook은 검토용 초안이며, 아래 태그 설치는 공개 후 검증한다.


## 가져다 쓰는 코드: 호출 전에 읽기

[Complete API reference](https://github.com/lunalab-ai/recommender/blob/2026-fall-w02b/src/API.md)

### [baseline_recommendations · definition](https://github.com/lunalab-ai/recommender/blob/2026-fall-w02b/src/luna_recsys/baselines.py#L114)

W02A 앱의 평점수/영화평균/집단평균 목록을 공통 형식으로 반환한다.

ratings: user_id/movie_id/rating DataFrame. movies: movie_id/title/장르 표.
method: mean(기본)/count/group. users: 기본None, group에서 사용자속성필수.
group_col: 집단 열 기본sex; group_value: 선택집단 기본F, 문자열로 비교.
genre: 기본None(전체), 지정한 장르0/1열로 필터. min_ratings: 기본5, top_n:
기본10, 모두 양수. 입력 표는 변경하지 않는다.

반환 최대top_n행은 movie_id/title/mean_rating/rating_count/basis 열.
count는 개수→평균 내림차순→ID, mean/group은 mean_rating_recommendations의
동점 규칙을 따른다. group은 선택집단 관측만 집계하고 조건에 맞는 목록이
전혀 없으면 전체 사용자 영화평균 목록으로 전환해 basis에 표시한다.
개별 평점의 group→movie→global 대체와 다르다. 학습 이력 제외/holdout은
이 함수 자체에서 하지 않으며 W02B 성능표는 FourMethodRecommender로 계산한다.

### [prepare_movielens · definition](https://github.com/lunalab-ai/recommender/blob/2026-fall-w02b/src/luna_recsys/datasets.py#L62)

캐시/다운로드/명시한 사본을 준비하고 데이터 종류까지 반환한다.

cache_dir: 폴더 문자열 또는 Path, 기본 data/local. 다운로드 파일 저장 위치.
local_dir: 기본None. 이미 가진 전체3파일의 폴더를 명시할 때만 사용한다.
mode: auto(기본)는 다운로드 실패 때 합성 대안을 명확히 표시한다. real은
실제 자료 준비 실패를 오류로 전달한다. synthetic은 다운로드 없이 가상
자료를 생성한다. 명시한 잘못된 local_dir는 합성으로 대체하지 않는다.
timeout: 네트워크 요청 제한 초, 기본20, 양수. 전체 함수 총시간 상한은 아니다.

반환 PreparedMovieLens의 data.users/data.movies/data.ratings는 표이며
mode/description은 출처를 설명한다. 실제 전체 자료는943명/1682편/100000평점.
폴더 생성과 다운로드/캐시 사용이 부작용이다. 실제 비교 실습은 반드시
prepare_movielens("data/local", mode="real")로 호출한다.
설정 오류는 ValueError, real 다운로드 실패는 OSError 등으로 보고한다.

### [build_baseline_lab · definition](https://github.com/lunalab-ai/recommender/blob/2026-fall-w02b/src/luna_recsys/demo_app.py#L106)

W02A 추천·평가 탭을 가진 Gradio Blocks를 반환한다.

dataset: MovieLens100K(users/movies/ratings 표). data_mode: 화면의 실제
데이터 설명문자열. 기존 인기목록의 장르/최소수/Top-N과 집단선택, 공통
holdout의 평균모형RMSE평가를 연결한다. 앱 준비 시 split_ratings(seed42)를
사용한다. 전체 앱 객체를 반환하며 launch는 호출자가 수행한다.
목록필터와 개별평점대체의 규칙은 각 탭의basis/level 설명을 따른다.
서버네트워크나데이터다운로드를 이 함수 자체에서 시작하지 않는다.

### [split_ratings · definition](https://github.com/lunalab-ai/recommender/blob/2026-fall-w02b/src/luna_recsys/evaluation.py#L28)

ratings의 행 위치를 한 번 나누어 RatingSplit(train,test,method)를 반환한다.

ratings: user_id/movie_id/rating을 포함한 관측 DataFrame, 최소2행.
test_size: 평가 비율0과1사이(기본0.25), seed: 난수 정수(기본42).
가능하면 user_id로 층화한다. 너무 작은 표의 대체는 random-small-data로
명시한다. 출력은 원본 인덱스/열을 보존하는 사본이며 원본을 변경하지 않는다.
MovieLens100K의 기본 설정은75000/25000이다. 같은 사용자가 양쪽에 존재할
수 있는 무작위 관측평가이며 시간분할/새 사용자평가가 아니다.
중복 관측쌍은 분할 전 전체 로더에서 검사한다. 비율/행 수 오류는 ValueError.

### [evaluate_means · definition](https://github.com/lunalab-ai/recommender/blob/2026-fall-w02b/src/luna_recsys/evaluation.py#L60)

동일한 split에서 세 평균 회귀모형을 학습·평가해 DataFrame을 반환한다.

split: RatingSplit의 train/test. users: 사용자ID와 집단속성 표.
group_col: 기본sex, min_group_ratings: 기본1인 최소 집단×영화 표본.
전체/영화/집단모형을 각각train에fit하고 test의ID로 예측한 뒤 test평점과
RMSE를 계산한다. 출력 열 method/rmse/test_count/group_used/movie_used/
global_used; 대체 건수의 합은 각 행의test_count다. 반올림 전에 평가하며
입력이나분할을변경하지않는다. 낮은RMSE가 순위만족도 개선을보증하지않는다.

### [MeanRatingPredictor · definition](https://github.com/lunalab-ai/recommender/blob/2026-fall-w02b/src/luna_recsys/rating_models.py#L29)

관측 평점을 이용해 범주별 상수 회귀모형을 학습하는 클래스.

Parameters
----------
mode : str, default 'movie'
    'global': 모든 행에 같은 학습 전체 평균. 'movie': 영화별 학습 평균.
    'group': 사용자의 집단과 영화가 같은 학습 행의 평균.
group_col : str, default 'sex'
    mode='group'일 때 users에서 읽는 집단 속성 열. 'occupation'도 가능.
min_group_ratings : int, default 1
    집단×영화 평균의 최소 학습 표본 수. 부족하면 영화 평균으로 대체.

학습은 L(theta)=sum((rating-theta)**2)를 최소화하는 theta=평균을
계산하는 것이다. 정답 rating을 쓰는 지도학습이며 경사하강법은 불필요하다.
생성자는 설정만 보관하고 fit이 통계 모수를 추정한다.

Attributes after fit
--------------------
global_mean_ : float, 학습 전체 평균.
movie_means_ : Series, movie_id가 인덱스인 영화별 평균.
user_groups_ : Series, group 모드의 user_id→집단 매핑.
group_means_ : Series, (집단, movie_id) 다중 인덱스 평균; 최소 표본 충족만.
밑줄 접미사는 학습 후 생기는 속성이라는 수업 코드의 명명 관례다.

Example
-------
model = MeanRatingPredictor('group', group_col='occupation', min_group_ratings=2)
model.fit(train, users)
y_hat = model.predict(test[['user_id', 'movie_id']])

집단 평균→영화 평균→전체 평균 순으로 대체하므로 새 집단/영화도 처리한다.
원본 표를 나중에 수정해도 이미 학습한 통계는 바뀌지 않는다. 재학습하려면
fit을 다시 호출한다. predict는 test 정답을 계산에 쓰지 않는다.

### [MeanRatingPredictor.fit · definition](https://github.com/lunalab-ai/recommender/blob/2026-fall-w02b/src/luna_recsys/rating_models.py#L77)

ratings의 정답 평점으로 평균 모수를 추정하고 self를 반환한다.

ratings: 학습 N행 DataFrame(user_id/movie_id/rating 필수). test를 넣지
않는다. users: 기본 None; group 모드에서는 user_id와 group_col을 가진
사용자 메타데이터가 필수다. ID는 유일하고 결측이 없어야 하며 ratings의
모든 사용자에 집단 정보가 있어야 한다. 다른 모드에서는 users를 무시한다.

1. 전체 rating 평균을 global_mean_에 저장한다.
2. movie_id별 rating 평균을 movie_means_에 저장한다.
3. group 모드는 user_id로 속성을 연결해 (집단,movie_id)별 평균/개수를
   구하고 min_group_ratings 이상인 평균만 group_means_에 저장한다.
실제 코드는 입력 검사를 먼저 끝낸 뒤 이 통계를 갱신한다. 입력 표는
수정하지 않는다. 반환 self이므로 MeanRatingPredictor().fit(train)처럼
연결 호출할 수 있다. 잘못된 표/집단 매핑에는 ValueError가 발생한다.

### [MeanRatingPredictor.predict_details · definition](https://github.com/lunalab-ai/recommender/blob/2026-fall-w02b/src/luna_recsys/rating_models.py#L115)

pairs의 각 사용자–영화 쌍에 예측값과 실제 사용 기준을 반환한다.

pairs: DataFrame, user_id/movie_id 필수. rating 열은 필요 없으며 있어도
사용하지 않는다. 반환은 같은 행 수·순서·인덱스의 DataFrame이며
prediction(float,1–5)과 level('group'/'movie'/'global') 열을 갖는다.
학습이 된 경우 빈 입력도 빈 출력을 반환한다. 원자료나 학습 상태는 수정하지
않는다. fit 전 또는 필수 열 누락은 ValueError.

먼저 전체 평균으로 배열을 채운다. 영화 평균이 있으면 덮어쓰고, group
모드에서 최소 표본을 충족하는 집단×영화 평균이 있으면 다시 덮어쓴다.
이 순서가 집단→영화→전체 대체 규칙을 구현한다. 평균은 반올림하지 않는다.
level을 세면 집단 모형이 실제로 집단 평균을 쓴 비율을 알 수 있다.

### [MeanRatingPredictor.predict · definition](https://github.com/lunalab-ai/recommender/blob/2026-fall-w02b/src/luna_recsys/rating_models.py#L147)

pairs(user_id/movie_id 표)의 평점 예측 Series를 반환한다.

predict_details(pairs)의 prediction 열만 꺼낸 편의 메소드다. 출력 길이는
입력 행 수이고 순서와 중복 인덱스까지 보존한다. 학습 후 여러 번 호출할 수
있고 상태를 갱신하지 않는다. test 정답과의 RMSE/MAE는 호출자가 별도로
계산한다. fit 전/필수 열 누락은 predict_details와 같은 ValueError.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_VERSION = "2026-fall-w02b"
if IN_COLAB:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        f"luna-recommender-course[apps] @ git+https://github.com/lunalab-ai/recommender.git@{COURSE_VERSION}",
    ], check=True)
    print("설치한 수업 버전:", COURSE_VERSION)
else:
    print("로컬 패키지로 검증합니다. 실제 Colab 실행 확인과는 별개입니다.")


## 2. 데이터 자동 준비와 모드 확인
`prepare_movielens`는 캐시를 재사용하거나 공식 서버에서 다운로드한다. 공식 서버 연결 실패 시
동일한 ZIP의 고정된 HTTPS 대체 경로를 사용하고 SHA-256과 MD5를 확인한다.
기본 `real` 모드는 실제 데이터만 사용한다. 모든 다운로드가 실패하면 원인을 표시하고 중단한다.
별도 파일 업로드는 필요 없다. `prepared.data`에는 세 DataFrame이 들어 있다.
합성 결과와 실제 MovieLens 결과는 데이터 모드로 구분한다.
`RECOMMENDER_DATA_DIR`은 로컬 검증자가 이미 가진 사본을 선택할 때만 사용하는 선택 설정이다.


In [ ]:
import pandas as pd
from luna_recsys import prepare_movielens, split_ratings, evaluate_means, MeanRatingPredictor
from luna_recsys.baselines import baseline_recommendations
from luna_recsys.demo_app import build_baseline_lab

cache_dir = Path("/content/recommender-data") if IN_COLAB else Path.home() / ".cache/luna-recsys"
prepared = prepare_movielens(
    cache_dir,
    local_dir=os.getenv("RECOMMENDER_DATA_DIR") or None,
    mode=os.getenv("RECOMMENDER_DATA_MODE", "real"),
)
dataset = prepared.data
users, movies, ratings = dataset.users, dataset.movies, dataset.ratings
print(prepared.description)
print(pd.DataFrame({"표": ["users", "movies", "ratings"], "행 수": [len(users), len(movies), len(ratings)]}).to_string(index=False))
print("사용자 표 열:", users.columns.tolist())
print("평점 표 열:", ratings.columns.tolist())
assert ratings["rating"].between(1, 5).all()


### 문제 1 · 연결 키 오류 해결
목표: 두 표의 공통 식별자를 찾는다. 아래는 **독자적인 장난감 표**다.
잘못된 `item` 키로 합치면 어떤 오류가 발생하는지 읽은 뒤 `join_key`를 고친다.
힌트: 왼쪽과 오른쪽에 모두 있는 열을 찾는다.
자가 점검: 연결 후 3행이며 `region`의 결측이 없어야 한다.


In [ ]:
toy_events = pd.DataFrame({"person": [1, 1, 2], "item": ["A", "B", "A"], "score": [3, 5, 4]})
toy_people = pd.DataFrame({"person": [1, 2], "region": ["east", "west"]})
try:
    toy_events.merge(toy_people, on="item")
except KeyError as error:
    print("관찰할 오류:", type(error).__name__, "— 두 표의 열 목록을 비교하세요.")

join_key = None  # 공통 키 이름으로 채우세요.
if join_key is None:
    print("문제 1 미완성: 키를 정하고 다시 실행하세요. 다음 실습은 계속할 수 있습니다.")
else:
    exercise_joined = toy_events.merge(toy_people, on=join_key, how="left", validate="many_to_one")
    assert len(exercise_joined) == 3 and exercise_joined["region"].notna().all()
    print("연결 검사 통과")


## 3. 인기 기준 비교
같은 최소 평점 수와 Top-N에서 정렬 기준만 바꾼다.
반환된 `rating_count`는 관측 평점 개수이고 `mean_rating`은 그 평균이다.
사용자별 원자료 대신 집계된 영화 결과만 출력한다.


In [ ]:
count_list = baseline_recommendations(ratings, movies, method="count", min_ratings=5, top_n=5)
mean_list = baseline_recommendations(ratings, movies, method="mean", min_ratings=5, top_n=5)
display(count_list)
display(mean_list)


### 문제 2 · 실행 전 예측하고 비교하기
최소 평점 수를 높이면 후보 수가 늘어날 수 있는지, 상위 영화는 같을지 먼저 적는다.
목표: 후보 필터와 순위 변화를 구분한다. 힌트: 후보는 줄어도 그 안의 순위 구성은 바뀔 수 있다.
자가 점검: 표시된 행 수가 최소 조건을 만족하며 결과가 비어 있는 경우도 설명한다.


In [ ]:
my_prediction = "여기에 실행 전 예측을 적으세요."
for threshold in [1, 20, 100]:
    comparison = baseline_recommendations(ratings, movies, method="mean", min_ratings=threshold, top_n=5)
    print("최소 평점 수:", threshold, "표시한 영화 수:", len(comparison))
    display(comparison)


## 4. 집단별 추천과 작은 집단
교재의 성별 그룹에서 출발해 직업 그룹으로 확장한다. 같은 집단이라고 취향이 같지는 않다.
`basis`는 실제 사용한 기준이다. 집단에서 조건에 맞는 영화가 전혀 없으면 전체 평균 목록으로
전환하므로, 이 경우 평점 수도 전체 사용자 기준으로 해석한다.


In [ ]:
student_list = baseline_recommendations(
    ratings, movies, method="group", users=users,
    group_col="occupation", group_value="student", min_ratings=5, top_n=5,
)
display(student_list)


### 문제 3 · 두 키 집계 빈칸 채우기
목표: 영화 하나로 묶는 것과 “집단×영화”로 묶는 것을 구분한다.
독립적인 지역·상품 데이터에서 `group_keys`와 `average_function`을 채운다.
힌트: `groupby`는 열 이름의 리스트를 받는다. 평균 집계의 함수 이름을 떠올린다.
자가 점검: 결과에 묶음 3개, 각 평균은 1–5 범위, 각 표본 수는 1 이상이어야 한다.


In [ ]:
group_toy = pd.DataFrame({
    "region": ["east", "east", "east", "west", "west"],
    "product": ["A", "A", "B", "A", "A"], "score": [3, 5, 2, 1, 5],
})
group_keys = None  # 두 집계 키를 리스트로 작성하세요.
average_function = None  # 평균 집계 함수 이름을 문자열로 작성하세요.
if group_keys is None or average_function is None:
    print("문제 3 미완성: 두 키와 집계 함수를 채우세요.")
else:
    exercise_stats = group_toy.groupby(group_keys)["score"].agg(average=average_function, n="count")
    assert len(exercise_stats) == 3 and exercise_stats["average"].between(1, 5).all()
    assert exercise_stats["n"].ge(1).all()
    display(exercise_stats)


## 5. 같은 학습·평가 분할 준비
`split_ratings`는 평점 행을 나누고 `RatingSplit`에 두 표와 분할 방식을 보관한다.
기본은 사용자 층화 75/25·seed 42다. 작은 데이터에서 조건이 부족하면 단순 무작위 분할임을 표시한다.
모든 예측기는 동일한 train으로 평균을 계산하고 동일한 test의 평점을 예측해야 한다.


In [ ]:
split = split_ratings(ratings, test_size=0.25, seed=42)
print("분할 방식:", split.method)
print("학습/평가 행 수:", len(split.train), len(split.test))
assert len(split.train) + len(split.test) == len(ratings)


### 문제 4 · 데이터 누수 디버깅
목표: 문법상 실행되더라도 잘못된 평가 코드를 찾아 고친다.
아래 문자열의 `ratings`는 전체 데이터다. 이 코드의 문제를 설명한 뒤 `fit_source`를 채운다.
힌트: `split`의 두 속성 중 모델이 평균을 배워도 되는 것은 하나다.
자가 점검: fit_source가 학습 표인지 확인한다. 문제용 객체는 아래 평가에 사용하지 않는다.


In [ ]:
buggy_code = 'leaky_model = MeanRatingPredictor("movie").fit(ratings)'
print("검토할 코드 (실행하지 않음):", buggy_code)
fit_source = None  # 평균을 학습해도 되는 DataFrame을 넣으세요.
if fit_source is None:
    print("문제 4 미완성: 전체 데이터의 평가 평점이 섞이지 않도록 고치세요.")
else:
    assert fit_source is split.train, "학습 데이터만 전달해야 합니다."
    exercise_model = MeanRatingPredictor("movie").fit(fit_source)
    print("학습 데이터 선택 확인")


## 6. RMSE를 손으로 계산하고 함수와 비교하기
실제값에서 예측값을 빼고, 제곱의 평균에 제곱근을 취한다. 평점 수 순위 점수는 평점 예측값이 아니다.
`root_mean_squared_error`에는 아래처럼 같은 순서의 실제 평점과 예측 평점을 전달한다.


In [ ]:
import numpy as np
from sklearn.metrics import root_mean_squared_error

true_example = np.array([5, 2, 4])
prediction_example = np.array([4, 2, 2])
errors = true_example - prediction_example
manual_rmse = np.sqrt(np.mean(errors ** 2))
library_rmse = root_mean_squared_error(true_example, prediction_example)
print("수계산 / 함수:", manual_rmse, library_rmse)
assert np.isclose(manual_rmse, library_rmse)


## 7. 공통 예측기와 실제 평가
`fit`은 학습 통계를 보관하고 `predict`는 ID 쌍에서 평점을 예측한다.
집단×영화 평균이 없거나 표본이 부족하면 학습 영화 평균, 그다음 학습 전체 평균을 사용한다.
표에는 각 방법이 같은 평가 행 수를 썼는지, 집단·영화·전체 평균을 몇 번 사용했는지도 나온다.


In [ ]:
model = MeanRatingPredictor("group", group_col="sex", min_group_ratings=1).fit(split.train, users)
predictions = model.predict(split.test[["user_id", "movie_id"]])
assert len(predictions) == len(split.test) and predictions.between(1, 5).all()
scores = evaluate_means(split, users, group_col="sex")
display(scores.round(4))


### 문제 5 · 작은 집단의 대체와 비교
목표: 세분화와 표본 수의 상충 관계를 관찰한다.
직업별 평균의 최소 표본 수를 1과 10으로 바꾸기 전에 RMSE와 `group_used` 변화를 예측한다.
힌트: 대체가 많아지는 것과 오차가 커지는 것은 같은 명제가 아니다.
자가 점검: 두 실행의 test_count가 같고 group/movie/global 사용 건수 합이 평가 행 수와 같다.


In [ ]:
for support in [1, 10]:
    comparison_scores = evaluate_means(split, users, group_col="occupation", min_group_ratings=support)
    assert (comparison_scores[["group_used", "movie_used", "global_used"]].sum(axis=1) == len(split.test)).all()
    print("최소 집단 표본 수:", support)
    display(comparison_scores.round(4))


## 8. 재현 가능한 결과 시각화
아래 막대의 숫자는 현재 데이터에서 직접 계산한 RMSE다. 제목의 모드를 함께 확인한다.
한국어 글꼴 설치가 필요하지 않도록 그래프 축은 영어로 쓰며 강의 설명은 한국어로 제공한다.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(["Global mean", "Movie mean", "Group/movie mean"], scores["rmse"], color=["#718096", "#319795", "#dd6b20"])
ax.bar_label(bars, fmt="%.4f", padding=3)
ax.set(ylabel="RMSE (lower is better)", title=f"{prepared.mode} | seed=42 | {split.method}")
ax.set_ylim(0, float(scores["rmse"].max()) * 1.2)
fig.tight_layout()
plt.show()


## 9. 지난 앱에 새 알고리즘 연결
builder는 화면과 callback을 연결하고, `launch`가 서버를 실행한다.
기존 장르·최소 평점 수·Top-N을 유지하면서 추천 기준과 집단을 바꿔 본다.
휴대폰에서는 카드와 탭을 사용하고 긴 표는 표 내부를 가로로 이동한다.
Colab의 공유 링크는 실행 중인 런타임에 연결된다. 종료하면 링크도 계속 사용할 수 없다.
로컬 일괄 검증에서는 화면 객체만 만들며 실제 브라우저 동작 검사는 별도로 수행한다.


In [ ]:
app = build_baseline_lab(dataset, data_mode=prepared.description)
if IN_COLAB:
    app.launch(share=True)
else:
    print("앱 객체 구성 완료. 로컬 브라우저 검증은 별도로 실행합니다.")
    app.close()


## 점검 퀴즈
1. 평점 표에서 한 행은 무엇을 의미하는가?
2. 평점 수가 가장 많은 영화가 평균 평점도 가장 높다고 할 수 있는가?
3. 최소 평점 수 조건을 높이면 후보 수는 어떻게 변할 수 있는가?
4. 사용자 집단별 영화 평균에는 어떤 두 집계 키가 필요한가?
5. 평가 데이터를 포함해 영화 평균을 계산하면 왜 문제가 되는가?
6. 학습 데이터에 없는 영화의 평점을 이 수업의 모델은 어떻게 예측하는가?
7. 평점 수 순위 점수를 RMSE에 그대로 넣으면 안 되는 이유는 무엇인가?
8. 집단별 평균의 RMSE가 더 작으면 모든 사용자의 추천 만족도가 높아졌다고 말할 수 있는가?

## 핵심 정리
데이터 모드와 키 확인 → 평균과 개수 비교 → 집단 및 표본 부족 해석 → train-only 평가 → 누적 앱.
문제를 다 풀지 못했다면 빈칸·오류 셀을 다시 실행하고 힌트와 자가 점검을 사용한다.
공통 구현은 `src/luna_recsys`에 누적하고 이 notebook에는 실험과 설명을 남긴다.

## 참고자료
- 임일, 『AI 에이전트를 위한 개인화 추천 알고리즘』, 청람, 2025, 2.2–2.4, pp.16–26.
- [MovieLens 100K](https://grouplens.org/datasets/movielens/100k/) — 데이터 출처.
- [pandas 집계](https://pandas.pydata.org/docs/getting_started/intro_tutorials/06_calculate_statistics.html) — 평균·개수·groupby.
- [pandas 결합](https://pandas.pydata.org/docs/getting_started/intro_tutorials/08_combine_dataframes.html) — merge의 키.
- [데이터 누수](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage) — 학습과 평가 분리.
- [RMSE API](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html).
- [Gradio 배치](https://gradio.app/guides/controlling-layout).
예제·연습·대체 규칙·앱은 수업용 추가 구성이다. 원자료와 사용자별 인구통계 행을 제출물에 포함하지 않는다.


In [ ]:
assert (scores.test_count == len(split.test)).all()
assert scores.rmse.between(0,4).all()
assert len(predictions) == len(split.test)
